In [1]:
import json
import os
import pandas as pd
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

print("🚀 Loading CLIP Bouncer Model...")
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def is_pie_chart(image_path):
    """Uses CLIP to instantly detect if an image is a pie chart."""
    try:
        image = Image.open(image_path).convert("RGB")
        labels = ["a photo of a pie chart", "a photo of a bar chart, line chart, or scatter plot"]
        inputs = clip_processor(text=labels, images=image, return_tensors="pt", padding=True).to(device)
        
        with torch.no_grad():
            outputs = clip_model(**inputs)
            probs = outputs.logits_per_image.softmax(dim=1).cpu().numpy()[0]
        
        return probs[0] > probs[1] # Returns True if it thinks it's a pie chart
    except Exception:
        return False # If image is corrupted, just let the next steps handle/skip it

def process_dataset_split(split_name, figureqa_folder_name):
    print(f"\n{'='*50}")
    print(f"🔥 STARTING SPLIT: {split_name.upper()}")
    print(f"{'='*50}")

    # --- 1. CONFIGURATION PATHS ---
    # Adapting paths based on the split name!
    figureqa_dir = f"./../FigureQA_Dataset/{figureqa_folder_name}"
    chartqa_dir = f"./../ChartQA_Dataset/{split_name}"
    plotqa_dir = f"./../PlotQA_Dataset/{split_name}"

    master_dataset = []

    # --- 2. PARSE FIGUREQA ---
    print(f"\n--- Parsing FigureQA ({split_name}) ---")
    if not os.path.exists(figureqa_dir):
        print(f"❌ ERROR: Could not find folder -> {figureqa_dir}")
    else:
        anno_path = os.path.join(figureqa_dir, 'annotations.json')
        if not os.path.exists(anno_path):
            print(f"❌ ERROR: Found folder, but no annotations.json -> {anno_path}")
        else:
            with open(anno_path, 'r') as f:
                figureqa_data = json.load(f)
            
            found_count = 0
            for item in figureqa_data:
                img_path = os.path.join(figureqa_dir, "png", f"{item['image_index']}.png")
                if not os.path.exists(img_path): continue
                    
                if item['type'] in ['vbar_categorical', 'hbar_categorical', 'line', 'dot_line']:
                    markdown_str = "TITLE | Unknown <0x0A> X_Label | Y_Label"
                    for model in item['models']:
                        x_values = model.get('x', [])
                        y_values = model.get('y', [])
                        if not isinstance(x_values, list): x_values = [x_values]
                        if not isinstance(y_values, list): y_values = [y_values]
                        for i in range(min(len(x_values), len(y_values))):
                            try:
                                y_val = round(float(y_values[i]), 2)
                            except (TypeError, ValueError):
                                y_val = y_values[i]
                            markdown_str += f" <0x0A> {x_values[i]} | {y_val}"
                            
                    master_dataset.append({"image_path": img_path, "target_text": markdown_str})
                    found_count += 1
            print(f"✅ Successfully extracted {found_count} charts from FigureQA!")

    # --- 3. PARSE CHARTQA ---
    print(f"\n--- Parsing ChartQA ({split_name}) ---")
    if not os.path.exists(chartqa_dir):
        print(f"❌ ERROR: Could not find folder -> {chartqa_dir}")
    else:
        chartqa_img_dir = os.path.join(chartqa_dir, "png")
        chartqa_csv_dir = os.path.join(chartqa_dir, "tables")
        
        if not os.path.exists(chartqa_img_dir) or not os.path.exists(chartqa_csv_dir):
             print(f"❌ ERROR: Missing 'png' or 'tables' subfolder inside {chartqa_dir}")
        else:
            found_count = 0
            skipped_pies = 0
            for img_file in os.listdir(chartqa_img_dir):
                if img_file.endswith(".png"):
                    base_name = img_file.replace(".png", "")
                    img_path = os.path.join(chartqa_img_dir, img_file)
                    csv_path = os.path.join(chartqa_csv_dir, f"{base_name}.csv")
                    
                    if os.path.exists(csv_path):
                        # ⚡ THE BOUNCER: Check if it's a Pie Chart!
                        if is_pie_chart(img_path):
                            skipped_pies += 1
                            continue
                            
                        try:
                            df = pd.read_csv(csv_path)
                            headers = " | ".join(df.columns)
                            markdown_str = f"TITLE | Unknown <0x0A> {headers}"
                            for index, row in df.iterrows():
                                row_str = " | ".join([str(val) for val in row.values])
                                markdown_str += f" <0x0A> {row_str}"
                                
                            master_dataset.append({"image_path": img_path, "target_text": markdown_str})
                            found_count += 1
                        except Exception:
                            pass # Skip corrupted CSVs
            print(f"✅ Extracted {found_count} Bar/Line charts. 🚫 Bouncer skipped {skipped_pies} Pie charts!")

    # --- 4. PARSE PLOTQA ---
    print(f"\n--- Parsing PlotQA ({split_name}) ---")
    if not os.path.exists(plotqa_dir):
        print(f"❌ ERROR: Could not find folder -> {plotqa_dir}")
    else:
        anno_path = os.path.join(plotqa_dir, 'annotations.json')
        if not os.path.exists(anno_path):
            print(f"❌ ERROR: Found folder, but no annotations.json -> {anno_path}")
        else:
            with open(anno_path, 'r') as f:
                plotqa_data = json.load(f)

            found_count = 0
            for item in plotqa_data:
                img_path = os.path.join(plotqa_dir, "png", f"{item['image_index']}.png")
                if not os.path.exists(img_path): continue
                    
                if item['type'] in ['vbar_categorical', 'hbar_categorical', 'line', 'dot_line']:
                    markdown_str = "TITLE | Unknown <0x0A> X_Label | Y_Label"
                    for model in item['models']:
                        x_values = model.get('x', [])
                        y_values = model.get('y', [])
                        if not isinstance(x_values, list): x_values = [x_values]
                        if not isinstance(y_values, list): y_values = [y_values]
                        for i in range(min(len(x_values), len(y_values))):
                            try:
                                y_val = round(float(y_values[i]), 2)
                            except (TypeError, ValueError):
                                y_val = y_values[i]
                            markdown_str += f" <0x0A> {x_values[i]} | {y_val}"
                            
                    master_dataset.append({"image_path": img_path, "target_text": markdown_str})
                    found_count += 1
            print(f"✅ Successfully extracted {found_count} charts from PlotQA!")

    # --- 5. SAVE JSONL ---
    print(f"\n--- Finishing Up {split_name.upper()} ---")
    output_file = f"master_dataset_{split_name}.jsonl"
    print(f"Saving {len(master_dataset)} total charts to {output_file}...")

    with open(output_file, 'w') as f:
        for entry in master_dataset:
            f.write(json.dumps(entry) + '\n')

    if len(master_dataset) > 0:
        print(f"✅ {split_name.upper()} Dataset Created Successfully!")
    else:
        print(f"⚠️ {split_name.upper()} Dataset is empty.")


# ==========================================
# EXECUTE FOR BOTH SPLITS
# (FigureQA uses 'train1' and 'val1' for folder names)
# ==========================================
# process_dataset_split(split_name="train", figureqa_folder_name="train1")
process_dataset_split(split_name="val", figureqa_folder_name="validation1")

/Users/aryangahlot/.pyenv/versions/tfenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Loading CLIP Bouncer Model...


Loading weights: 100%|███████████████████████████████████████████████████| 398/398 [00:00<00:00, 1938.79it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 



🔥 STARTING SPLIT: VAL

--- Parsing FigureQA (val) ---
✅ Successfully extracted 16000 charts from FigureQA!

--- Parsing ChartQA (val) ---
✅ Extracted 946 Bar/Line charts. 🚫 Bouncer skipped 110 Pie charts!

--- Parsing PlotQA (val) ---
✅ Successfully extracted 33650 charts from PlotQA!

--- Finishing Up VAL ---
Saving 50596 total charts to master_dataset_val.jsonl...
✅ VAL Dataset Created Successfully!


In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "2" # Limits CPU core usage
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0" # Prevents RAM hoarding
import time # We need this for the cool-down hack

In [4]:
import json
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
import shutil

# --- 1. CONFIGURATION ---
SPLIT = "val"  # Change this to "val" and run again for validation data

# Update these paths to point to your local dataset split folders
# Note: FigureQA often uses 'train1' and 'val1'
datasets = {
    "FigureQA": f"./../FigureQA_Dataset/{SPLIT}1",
    "ChartQA": f"./../ChartQA_Dataset/{SPLIT}",
    "PlotQA": f"./../PlotQA_Dataset/{SPLIT}"
}

# Create YOLO-standard directory structure
output_images_dir = f"./yolo_dataset/images/{SPLIT}"
output_labels_dir = f"./yolo_dataset/labels/{SPLIT}"
os.makedirs(output_images_dir, exist_ok=True)
os.makedirs(output_labels_dir, exist_ok=True)

CLASS_BAR = 0

def process_and_copy(img_path, bboxes, out_img_dir, out_lab_dir):
    """Normalizes labels AND copies the image to the YOLO folder"""
    if not os.path.exists(img_path): return
    
    try:
        with Image.open(img_path) as img:
            w_img, h_img = img.size
        
        yolo_lines = []
        for box in bboxes:
            if isinstance(box, dict):
                x, y, w, h = box['x'], box['y'], box['w'], box['h']
            else:
                x, y, w, h = box
                
            cx, cy = (x + w/2) / w_img, (y + h/2) / h_img
            nw, nh = w / w_img, h / h_img
            yolo_lines.append(f"{CLASS_BAR} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        if yolo_lines:
            base_name = os.path.basename(img_path)
            # Copy Image
            shutil.copy(img_path, os.path.join(out_img_dir, base_name))
            # Save Label
            txt_filename = base_name.rsplit('.', 1)[0] + ".txt"
            with open(os.path.join(out_lab_dir, txt_filename), 'w') as f:
                f.write("\n".join(yolo_lines))
    except: return

# --- 2. PROCESSING ---

# A. FIGUREQA & PLOTQA
for name in ["FigureQA", "PlotQA"]:
    print(f"\nProcessing {name} [{SPLIT}]...")
    base_dir = datasets[name]
    anno_path = os.path.join(base_dir, "annotations.json")
    
    if os.path.exists(anno_path):
        with open(anno_path, 'r') as f:
            data = json.load(f)
        for item in tqdm(data, desc=f"Extracting {name}"):
            if item['type'] in ['vbar_categorical', 'hbar_categorical']:
                img_path = os.path.join(base_dir, "png", f"{item['image_index']}.png")
                all_boxes = []
                for model in item['models']:
                    if 'bboxes' in model: all_boxes.extend(model['bboxes'])
                process_and_copy(img_path, all_boxes, output_images_dir, output_labels_dir)

# B. CHARTQA
print(f"\nProcessing ChartQA [{SPLIT}]...")
chart_base = datasets["ChartQA"]
chart_anno_dir = os.path.join(chart_base, "annotations")

if os.path.exists(chart_anno_dir):
    anno_files = [f for f in os.listdir(chart_anno_dir) if f.endswith('.json')]
    for filename in tqdm(anno_files, desc="Extracting ChartQA"):
        with open(os.path.join(chart_anno_dir, filename), 'r') as f:
            item = json.load(f)
        if item.get('type') in ['v_bar', 'h_bar', 'vbar_categorical', 'hbar_categorical']:
            img_path = os.path.join(chart_base, "png", filename.replace('.json', '.png'))
            all_boxes = []
            for model in item.get('models', []):
                if 'bboxes' in model: all_boxes.extend(model['bboxes'])
            process_and_copy(img_path, all_boxes, output_images_dir, output_labels_dir)

print(f"\n✅ YOLO {SPLIT.upper()} set generated!")


Processing FigureQA [val]...

Processing PlotQA [val]...


Extracting PlotQA: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 33650/33650 [00:13<00:00, 2548.64it/s]



Processing ChartQA [val]...


Extracting ChartQA: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 1056/1056 [00:00<00:00, 1551.81it/s]


✅ YOLO VAL set generated!


In [4]:
import json
import os
import shutil
from PIL import Image
from tqdm import tqdm

# Define datasets and their splits
# Adjust these paths if your validation folders are named differently
dataset_configs = [
    {"name": "FigureQA", "base": "./../FigureQA_Dataset/train1", "split": "train"},
    {"name": "FigureQA", "base": "./../FigureQA_Dataset/validation1", "split": "val"},
    {"name": "PlotQA", "base": "./../PlotQA_Dataset/train", "split": "train"},
    {"name": "PlotQA", "base": "./../PlotQA_Dataset/val", "split": "val"}
]

CLASS_BAR = 0  

def process_and_copy(img_path, bboxes, split):
    out_img_dir = f"./datasets/yolo_dataset_bar/images/{split}"
    out_lab_dir = f"./datasets/yolo_dataset_bar/labels/{split}"
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lab_dir, exist_ok=True)

    if not os.path.exists(img_path): return
    try:
        with Image.open(img_path) as img:
            w_img, h_img = img.size
        
        yolo_lines = []
        for box in bboxes:
            # Handle different coordinate formats
            if isinstance(box, dict):
                x, y, w, h = box['x'], box['y'], box['w'], box['h']
            else:
                x, y, w, h = box
                
            # Normalize to YOLO format (0-1)
            cx, cy = (x + w/2) / w_img, (y + h/2) / h_img
            nw, nh = w / w_img, h / h_img
            yolo_lines.append(f"{CLASS_BAR} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        if yolo_lines:
            base_name = os.path.basename(img_path)
            shutil.copy(img_path, os.path.join(out_img_dir, base_name))
            
            txt_filename = base_name.rsplit('.', 1)[0] + ".txt"
            with open(os.path.join(out_lab_dir, txt_filename), 'w') as f:
                f.write("\n".join(yolo_lines))
    except:
        pass

# Execute Strict Extraction for all configs
for config in dataset_configs:
    print(f"\nProcessing {config['name']} ({config['split']})...")
    anno_path = os.path.join(config['base'], "annotations.json")
    
    if os.path.exists(anno_path):
        with open(anno_path, 'r') as f:
            data = json.load(f)
            
        for item in tqdm(data):
            # STRICT FILTER: Only vbar and hbar
            if item['type'] in ['vbar_categorical', 'hbar_categorical']:
                img_path = os.path.join(config['base'], "png", f"{item['image_index']}.png")
                all_boxes = []
                for model in item['models']:
                    if 'bboxes' in model: 
                        all_boxes.extend(model['bboxes'])
                
                process_and_copy(img_path, all_boxes, config['split'])

print("\n✅ Clean Train & Val datasets generated! No more ChartQA noise.")


Processing FigureQA (train)...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100000/100000 [00:22<00:00, 4401.60it/s]



Processing FigureQA (val)...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20000/20000 [00:03<00:00, 5160.65it/s]



Processing PlotQA (train)...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 157070/157070 [00:07<00:00, 20954.73it/s]



Processing PlotQA (val)...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 33650/33650 [00:13<00:00, 2414.23it/s]


✅ Clean Train & Val datasets generated! No more ChartQA noise.
